# 连接网络**scikit-rf** 支持连接 N 端口网络中的任意端口。它通过一种称为子网络增长的算法来实现这一点[[1]](#参考文献)，该算法可通过 `connect()` 函数使用。请注意，此函数会考虑端口阻抗。如果两个连接的端口具有不同的端口阻抗，则会插入适当的阻抗失配。以下是一些常见的情况，用于说明此功能。

In [ ]:
import skrf as rf

## 串联 2 端口和 1 端口网络一个常见的问题是将两个网络连接在一起，也称为串联网络，从而创建一个新的网络。下图说明了几个简单的例子，其中端口号以灰色标识：<img src="figures/networks_connecting_2_2ports.svg" width="600">或者，<img src="figures/networks_connecting_2port_1port.svg" width="600">让我们通过将一条传输线（2 端口网络）连接到一个短路（1 端口网络）来创建一个延迟短路（1 端口网络），以此来说明：<img src="figures/networks_delay_short.svg" width="600">由于串联网络是一个常见的操作，因此可以通过 `**` 运算符或 `cascade` 函数方便地进行：

In [ ]:
line = rf.data.wr2p2_line  # 2-port
short = rf.data.wr2p2_short  # 1-port

delayshort = line ** short  # --> 1-port Network
print(delayshort)

或者，等效地，可以使用 `cascade()` 函数：

In [ ]:
delayshort2 = rf.network.cascade(line, short)
print(delayshort2 == delayshort)  # the result is the same

当然，可以使用 `connect()` 函数将两个 2 端口网络连接起来。`connect()` 函数需要指定要连接的网络以及端口号。在我们的示例中，线路的端口 1 连接到短路的端口 0：

In [ ]:
delayshort3 = rf.network.connect(line, 1, short, 0)
print(delayshort3 == delayshort)

通常需要将多个网络对象连接起来：<img src="figures/networks_connecting_N_2ports.svg" width="700">或者，<img src="figures/networks_connecting_N_2ports_1port.svg" width="700">这可以通过使用链式 `**` 运算符或方便的函数 `cascade_list` 来实现：

In [ ]:
line1 = rf.data.wr2p2_line  # 2-port
line2 = rf.data.wr2p2_line  # 2-port
line3 = rf.data.wr2p2_line  # 2-port
line4 = rf.data.wr2p2_line  # 2-port
short = rf.data.wr2p2_short  # 1-port

chain1 = line1 ** line2 ** line3 ** line4 ** short

chain2 = rf.network.cascade_list([line1, line2, line3, line4, short])

print(chain1 == chain2)

## 级联 2N 端口网络

级联运算符 `**` 也适用于 2N 端口网络，端口连接方式如下：<img src="figures/networks_connecting_2_2Nports.svg" width="600">它也适用于多个 2N 端口网络。例如，假设您想要级联三个 4 端口网络 `ntw1`、`ntw2` 和 `ntw3`，您可以使用：```resulting_ntw = ntw1 ** ntw2 ** ntw3```这在 [平衡网络示例](https://github.com/scikit-rf/scikit-rf/blob/master/examples/networktheory/Balanced%20Network%20De-embedding.ipynb) 中有所说明。

## 级联多端口网络为了在多端口网络之间建立特定的连接，有三种解决方案，这在很大程度上取决于要构建的电路的复杂程度：* 对于较少的连接：使用 `connect()` 函数。* 对于中等复杂度的连接，例如，您可能需要将多个网络并行连接到同一个交叉点，我们提供了 `parallelconnect()` 方法。此方法在 `connect()` 的简单性和 `Circuit` 对象的灵活性之间提供了一种平衡。有关更多信息，请参阅 [`paralleconnect()` 文档](../api/generated/skrf.network.paralleconnect.html#skrf.network.paralleconnect)。* 对于在许多任意 N 端口网络之间进行更高级的连接，`Circuit` 对象更相关，因为它允许显式地定义端口和网络之间的连接。有关更多信息，请参阅 [Circuit 文档](Circuit.ipynb)。例如，终止一个 3 端口网络的一个端口，例如理想的 3 路分配器：<img src="figures/networks_connecting_3port_1port.svg" width="600">可以这样实现：

In [ ]:
tee = rf.data.tee

将 T 型接头的端口 `1` 连接到延迟短路的端口 `0`。

In [ ]:
terminated_tee = rf.network.connect(tee, 1, delayshort, 0)
terminated_tee

`parallelconnect()` 方法也可以通过对语法进行轻微修改来处理这种情况。

In [ ]:
terminated_tee_par = rf.network.parallelconnect([tee, delayshort], [1, 0])
terminated_tee_par

在之前的示例中，3 端口网络 `tee` 的端口 #2 变成了结果 2 端口网络中的端口 #1。

## 多端口网络的多次连接在多次使用 `connect` 函数时，需要跟踪端口编号，这可能会比较繁琐（这也是为什么 [电路对象](Circuit.ipynb) 可能更易于使用）。让我们通过以下示例来说明：将一个 T 型连接器（3 端口）的端口 #1 连接到一条传输线的端口 #0（2 端口）：<img src="figures/networks_connecting_3port_2port.svg" width="600">为了在连接操作后跟踪端口方案，让我们更改端口特性阻抗（如图中红色部分所示）：

In [ ]:
tee.z0 = [1, 2, 3]
line.z0 = [10, 20]
# the resulting network is:
rf.network.connect(tee, 1, line, 0)

## 从交集角度看网络连接在之前的示例中，我们简要介绍了 `parallelconnect()` 方法。在本节中，我们将通过多个示例详细介绍 `parallelconnect()` 方法的应用场景，并提供比较三种解决方案的建议。首先，让我们考虑最简单的示例：在具有 N 个端口的网络中，将任意两个端口内部连接。这将产生一个具有 (N-2) 个端口的网络。<img src="figures/networks_innerconnect_2_port.svg" width="600">在这里，我们可以使用 `innerconnect()` 方法来连接 N 端口网络中的任意两个端口：```# 连接 N 端口网络中的第 m 个和第 n 个端口inner_network = rf.network.innerconnect(nport_network, m, n)````parallelconnect()` 方法可以处理类似上述的内部连接情况：```inner_network_par = rf.network.parallelconnect(nport_network, [[m, n]])```

如果预计需要内部连接超过 2 个端口，`innerconnect()` 方法将无法满足此需求。您需要考虑使用 `tee`（分路器）或 `Circuit` 对象来实现您的目标。但是，`parallelconnect()` 为此类情况提供了一个简单的解决方案。```# 一个内部连接 N 端口网络中端口列表的示例ports_list = [[m, n, ..., y, z]]inner_network2_par = rf.network.parallelconnect(nport_network, ports_list)```

第二个例子涉及连接两个多端口网络。由于这个案例已经通过之前的演示，我们不再在此深入探讨。接下来，我们考虑多个多端口`Networks`的并联连接。一个常见的应用是在构建`T型滤波器电路`中，如图所示，取自[electroniclinic.com](https://www.electroniclinic.com/filter-circuit-and-need-of-filters-in-electronics/#google_vignette)。![T型滤波器电路](figures/t_type_filter.png)让我们忽略具体的细节，并比较在这三个方法中实现该示例的方法，你可以使用：```# 1. 使用 tee 的 connect() 方法t_type_filter_ntwk = rf.network.connect(L1, 1, tee, 0)t_type_filter_ntwk = rf.network.connect(t_type_filter_ntwk, 1, C, 0)t_type_filter_ntwk = rf.network.connect(t_type_filter_ntwk, 2, L2, 0)t_type_filter_ntwk# 2. 电路对象方法cnx = [    [(port1, 0), (L1, 0)],    [(port2, 0), (C , 1)],    [(port3, 0), (L2, 1)],    [(L1, 1), (C, 0), (L2, 0)]]t_type_filter_ckt = rf.cicuit.Circuit(cnx)t_type_filter_ntwk = t_type_filter_ckt.networkt_type_filter_ntwk# 3. parallelconnect() 方法t_type_filter_ntwk = rf.network.parallelconnect([L1, C, L2], [1, 0, 0])t_type_filter_ntwk```可以看出，`parallelconnect()` 方法可以非常清晰和简洁地实现`T型滤波器`电路的构建。

在本节中，我们将比较 `parallelconnect()` 方法，并将其与 `connect()/innerconnect()` 方法以及用于连接 `Networks` 的 `Circuit` 对象进行比较，从交叉点的角度进行分析。`parallelconnect()` 方法在处理多个并行连接时表现出色，并且所需的步骤最少，因此非常适合需要简单性和效率的场景。相比之下，`connect()/innerconnect()` 方法更适合处理更简单、顺序连接的情况，而 `Circuit` 对象则用于处理更复杂、多层连接，并提供更大的控制。从交叉点的角度来看，`Circuit` 对象最适合管理具有多个交叉点的复杂网络，这是一项其他方法无法完成的任务，因为其他方法的设计目的是用于处理单个交叉点的情况。`innerconnect()/connect()` 方法的限制在于只能处理单个或成对 `Networks` 之间的连接，而 `parallelconnect()` 则消除了对 `Networks` 数量的限制，并且可以有效地在一个代码行中为多个 `Networks` 建立连接，因此对于在多个点具有并行连接的复杂电路而言，它具有特别的优势。

## References[1] Compton, R.C.; , "Perspectives in microwave circuit analysis," Circuits and Systems, 1989., Proceedings of the 32nd Midwest Symposium on , vol., no., pp.716-718 vol.2, 14-16 Aug 1989. URL: http://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=101955&isnumber=3167